In [1]:
import numpy as np
import pandas as pd
import timeit
from functools import partial
from copy import deepcopy
import sys
# Add measurement mcts python package to path
sys.path.append('../src/measurement_mcts')
from measurement_mcts.mcts.mcts import mcts_with_rollout
from measurement_mcts.mcts.tree_viz import render_pyvis
from measurement_mcts.state_evaluation.hertg import HERTG
from measurement_mcts.environment.measurement_control_env import MeasurementControlEnvironment

# Create the environment
env = MeasurementControlEnvironment(init_reset=False)

Toy Measurement Control Initialized


In [17]:
starting_state = env.reset()

In [16]:
def get_percent_done(state, env):
    # First calculate total trace at start
    total_trace = env.init_covariance_trace
    final_coner_trace = env.final_corner_cov_trace
    
    # Then calculate trace of ooi cov
    ooi_covs = state[2]
    per_ooi_traces = np.trace(ooi_covs, axis1=2, axis2=3)
    all_traces = per_ooi_traces.flatten()
    
    # Make any traces below the final corner trace 0
    all_traces[all_traces < final_coner_trace] = 0
    
    # Sum the traces and calculate the percentage
    sum_traces = np.sum(all_traces)
    return (1 - sum_traces / total_trace) * 100
    
get_percent_done(starting_state, env)

0.0

In [27]:
def get_mcts_metrics(state, object_true_state, max_actions=100, LI=100,
                     EF=0.1, DF=1.0, rollout_method='random_same',
                     hertg_method='static', rollout_pre_collision_stop=True) -> dict:
    """
    Run MCTS and return the metrics.
    params:
        state: the initial state of the environment
        object_true_state: the true state of the object
        max_actions: the maximum number of actions to take
        LI: the length of the interval for the HERTG method
        EF: the exploration factor for the HERTG method
        DF: the discount factor for the HERTG method
        rollout_method: the rollout method to use
        hertg_method: the HERTG method to use
        
    returns:
        metrics: a dictionary of metrics
    """
    
    # Create environment and HERTG objects
    env = MeasurementControlEnvironment(init_reset=False)
    env.object_manager.set_true_state(object_true_state)
    hertg = HERTG(state, env, method=hertg_method)
    
    # Create metric trackers
    cumulative_reward = 0.
    num_actions = 0
    done = False
    start_time = timeit.default_timer()
    for i in range(max_actions):
        # Run MCTS and take the best action
        root = mcts_with_rollout(env, state, LI, EF, DF, rollout_method,
                                 rollout_pre_collision_stop, hertg=hertg)
        best_action_idx = np.argmax(root.child_Q())
        state, reward, done = env.step(state, env.action_space[best_action_idx])
        
        # Reset the horizon to 0
        state_list = list(state)
        state_list[3] = 0
        state = tuple(state_list)
        
        # Increment the cumulative reward and number of actions
        cumulative_reward += reward
        num_actions = i + 1
        if done:
            break
        
    comp_time = timeit.default_timer() - start_time
    percent_done = get_percent_done(state, env)
    
    metrics = {
        'LI': LI,
        'EF': EF,
        'DF': DF,
        'rollout_method': rollout_method,
        'hertg_method': hertg_method,
        'done': done,
        'percent_done': percent_done,
        'cumulative_reward': cumulative_reward,
        'num_actions': num_actions,
        'computation_time': comp_time,
        'computation_per_action': comp_time / num_actions,
    }
    
    return metrics

In [29]:
get_mcts_metrics(env.get_state(), env.object_manager.get_true_state(), max_actions=200)

Toy Measurement Control Initialized


c:\Users\austi\Documents\CodeScratch\MeasurementMCTS\metrics\../src/measurement_mcts\measurement_mcts\mcts\mcts.py:222: RuntimeWarning: divide by zero encountered in log
  * np.sqrt(np.log(self.number_visits)


{'LI': 100,
 'EF': 0.1,
 'DF': 1.0,
 'rollout_method': 'random_same',
 'hertg_method': 'static',
 'done': True,
 'percent_done': 99.23445381978621,
 'cumulative_reward': 6.415937931625999,
 'num_actions': 153,
 'computation_time': 273.5229718000628,
 'computation_per_action': 1.787731841830476}